# AG983 | Class 1 Workshop
## Text Pre-Processing: Decisions and Consequences

**University of Strathclyde Business School** | Dr James Bowden

---

This workshop accompanies Section 6.4 of the Class 1 lecture. You will work through each stage of a text pre-processing pipeline, making decisions at each step and observing the consequences for your analysis.

The corpus is drawn from the Enron email dataset (Klimt & Yang, 2004) — released into the public domain by the Federal Energy Regulatory Commission during the 2001 Enron investigation. As discussed in the lecture, this corpus is well-suited to demonstrating pre-processing challenges because it combines the hedged, forward-looking register of corporate financial communication with the informal directness of internal correspondence.

**Instructions:** Run each cell in order using **Shift+Enter**. Make your choices where indicated. Record your observations in the accompanying worksheet.

> *Each decision you make is a methodological choice with real consequences for your analysis — as Renault (2020) shows, optimal pre-processing increases sentiment-returns correlation by ~55% relative to a naive baseline.*

---
## Step 0: Environment Setup

Run this cell to install dependencies and load the dictionaries. **Run this first.**

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "nltk", "pandas", "matplotlib", "seaborn", "wordcloud", "-q"],
               check=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import re
import warnings
from collections import Counter
warnings.filterwarnings("ignore")

# NLTK downloads
import nltk
for pkg in ["punkt", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

# ── Load corpus from GitHub ────────────────────────────────────────────────
CORPUS_URL = ("https://raw.githubusercontent.com/iamjamesbowden/AG983/"
              "main/class1/data/enron_sample.csv")
corpus = pd.read_csv(CORPUS_URL)

# ── Load dictionaries from AG952 repo ─────────────────────────────────────
LM_URL = ("https://raw.githubusercontent.com/iamjamesbowden/AG952/"
          "main/assignments/march2026/data/lm_dictionary.csv")
HV_URL = ("https://raw.githubusercontent.com/iamjamesbowden/AG952/"
          "main/assignments/march2026/data/harvard_iv_dictionary.csv")

lm  = pd.read_csv(LM_URL)
hiv = pd.read_csv(HV_URL)

LM_NEGATIVE = set(lm[lm["Negative"] == 1]["Word"].str.lower())
LM_POSITIVE = set(lm[lm["Positive"] == 1]["Word"].str.lower())
HV_NEGATIVE = set(hiv[hiv["Negative"] == 1]["Word"].str.lower())
HV_POSITIVE = set(hiv[hiv["Positive"] == 1]["Word"].str.lower())

# ── Stop-word lists ────────────────────────────────────────────────────────
STD_SW = set(stopwords.words("english"))

# Finance-adjusted: remove modal verbs, negation, directional terms
# (Loughran & McDonald, 2011; Renault, 2020)
FINANCE_PROTECTED = {
    # Modal verbs — forward-looking commitment and uncertainty signals
    "will", "may", "could", "should", "would", "must", "shall", "might", "can",
    # Negation — reverses meaning (e.g. "not material", "no guarantee")
    "not", "no", "nor", "never", "neither",
    # Directional terms — specific meaning in investment contexts
    "up", "down", "above", "below", "over", "under",
    # Other financially meaningful terms that NLTK removes
    "more", "less", "very", "against", "before", "after",
    "during", "between", "through", "further", "however",
    "although", "despite", "unless", "until",
}
FIN_SW = STD_SW - FINANCE_PROTECTED

# ── Helpers ────────────────────────────────────────────────────────────────
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# Consistent plot style
plt.rcParams.update({
    "figure.facecolor": "#0f0f1a",
    "axes.facecolor":   "#0f0f1a",
    "axes.edgecolor":   "#444",
    "axes.labelcolor":  "#ddd",
    "xtick.color":      "#aaa",
    "ytick.color":      "#aaa",
    "text.color":       "#ddd",
    "grid.color":       "#2a2a3a",
    "grid.linestyle":   "--",
    "grid.alpha":       0.4,
})

print(f"Corpus loaded:          {len(corpus):>4} emails")
print(f"  pre-crisis (2000):    {(corpus.period=='pre_crisis').sum():>4}")
print(f"  crisis (2001):        {(corpus.period=='crisis').sum():>4}")
print(f"  collapse (2002):      {(corpus.period=='collapse').sum():>4}")
print(f"\nLM Negative words:     {len(LM_NEGATIVE):>5}")
print(f"LM Positive words:     {len(LM_POSITIVE):>5}")
print(f"Harvard IV Negative:   {len(HV_NEGATIVE):>5}")
print(f"Harvard IV Positive:   {len(HV_POSITIVE):>5}")
print(f"\nStandard stop-words:  {len(STD_SW):>5}")
print(f"Finance-adjusted:      {len(FIN_SW):>5}  (protecting {len(FINANCE_PROTECTED)} finance terms)")
print("\n✓ Environment ready.")

---
## Step 1: Explore the Corpus

Before processing any text, understand what you are working with. Run the cells below to examine the corpus structure, email length distribution, and a sample of raw text.

*Record your observations for **Worksheet Question 1**.*

In [ ]:
# Corpus overview
print("=" * 55)
print("CORPUS OVERVIEW")
print("=" * 55)
print(corpus.groupby(["period","category"]).size().rename("emails").to_string())
print(f"\nTotal emails:  {len(corpus)}")
print(f"Total words:   {corpus.word_count.sum():,}")
print(f"Mean length:   {corpus.word_count.mean():.0f} words")
print(f"Shortest:      {corpus.word_count.min()} words")
print(f"Longest:       {corpus.word_count.max()} words")

# Length distribution by category
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("#0f0f1a")

COLORS = {"pre_crisis": "#3b82f6", "crisis": "#f59e0b", "collapse": "#ef4444"}
for ax, groupcol, title in [
    (axes[0], "period",   "Email length by time period"),
    (axes[1], "category", "Email length by category"),
]:
    for name, grp in corpus.groupby(groupcol):
        color = COLORS.get(name, "#8b5cf6")
        ax.hist(grp["word_count"], bins=12, alpha=0.6, label=name, color=color)
    ax.set_title(title, color="#ddd", pad=8)
    ax.set_xlabel("Word count")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=7)
plt.suptitle("Corpus structure — Enron email sample (Klimt & Yang, 2004)",
             color="#ddd", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Show a raw email — no processing applied
sample = corpus.iloc[5]
print(f"From:     {sample['sender']}")
print(f"Date:     {sample['date']}")
print(f"Period:   {sample['period']}")
print(f"Category: {sample['category']}")
print(f"Subject:  {sample['subject']}")
print(f"Words:    {sample['word_count']}")
print("-" * 60)
print(sample["body"])

---
## Step 2: Tokenisation

Tokenisation is the first step in converting raw text into units your model can count and compare.

Two common approaches:
- **Whitespace split** — fast, no library needed, but treats `"liability."` and `"liability"` as different tokens
- **NLTK word_tokenize** — handles punctuation correctly, recognises contractions

Run the cell below and compare the outputs on the same sentence from the corpus.

In [ ]:
text = corpus.iloc[3]["body"]   # Pick a representative email
print("RAW TEXT:")
print(text[:300])
print()

ws_tokens = text.split()
nl_tokens = word_tokenize(text.lower())

print(f"Whitespace split:    {len(ws_tokens)} tokens")
print(f"NLTK word_tokenize: {len(nl_tokens)} tokens")
print()

# Show tokens that differ between the two approaches
ws_set = set(t.lower() for t in ws_tokens)
nl_set = set(nl_tokens)
in_ws_not_nl = sorted(ws_set - nl_set)[:15]
in_nl_not_ws = sorted(nl_set - ws_set)[:15]

print("Tokens in whitespace split but NOT NLTK (punctuation attached):")
print("  ", in_ws_not_nl)
print()
print("Tokens in NLTK but NOT whitespace split (punctuation separated):")
print("  ", in_nl_not_ws[:10])

> **Worksheet Q2:** Why does it matter which tokenisation method you use before applying a sentiment dictionary? What could go wrong with whitespace splitting when using the Loughran-McDonald lexicon?

---
## Step 3: Case Folding and Punctuation Removal

**Case folding** converts all text to lowercase so that `"Revenue"` and `"revenue"` are treated as the same token.

**Punctuation removal** prevents the same word appearing as multiple tokens (`"liability."` vs `"liability"`).

The cells below let you compare token frequency distributions with and without these steps. Look at what changes — and what is lost.

In [ ]:
def get_tokens(texts, lowercase=True, remove_punct=True):
    """Tokenise a list of texts with configurable options."""
    all_tokens = []
    for text in texts:
        if lowercase:
            text = text.lower()
        tokens = word_tokenize(text)
        if remove_punct:
            tokens = [t for t in tokens if t.isalpha()]
        all_tokens.extend(tokens)
    return all_tokens

texts = corpus["body"].tolist()

# Four configurations
configs = {
    "Raw (whitespace)":           texts[0].split(),
    "Tokenised only":             get_tokens(texts[:10], lowercase=False, remove_punct=False),
    "Case-folded":                get_tokens(texts[:10], lowercase=True,  remove_punct=False),
    "Case-fold + punct removed":  get_tokens(texts[:10], lowercase=True,  remove_punct=True),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.patch.set_facecolor("#0f0f1a")

for ax, (label, tokens) in zip(axes, configs.items()):
    freq = Counter(tokens).most_common(15)
    words, counts = zip(*freq)
    bars = ax.barh(list(reversed(words)), list(reversed(counts)), color="#3b82f6", alpha=0.85)
    ax.set_title(label, color="#ddd", fontsize=9, pad=6)
    ax.set_xlabel("Count")
plt.suptitle("Effect of case folding and punctuation removal on token frequency",
             color="#ddd", y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

> **Note from the lecture (Renault, 2020):** retaining `!` and `?` improves financial sentiment classification by 0.3%. Not all punctuation is noise.

---
## Step 4: Stop-Word Removal

This is one of the most consequential decisions in preprocessing financial text.

The **standard NLTK** stop-word list removes 179 common English words — including all modal verbs (`will`, `may`, `could`, `should`, `would`, `must`) and all negation terms (`not`, `no`, `nor`).

As Loughran & McDonald (2011) identify, modal verbs are among the most informationally rich tokens in corporate disclosures:
- *"will"* — forward-looking commitment
- *"may"* — uncertainty signal
- *"could"* — conditional risk language
- *"not"* — negation: removing it would reverse meaning (e.g. *"not material"* → *"material"*)

The **finance-adjusted** list removes these terms from the stop list, preserving them as tokens.

Run the cell below to see the difference side by side, then choose your stop-word setting for the pipeline.

In [ ]:
texts = corpus["body"].tolist()
all_tokens = get_tokens(texts, lowercase=True, remove_punct=True)

# Three variants
no_sw_tokens  = all_tokens
std_tokens    = [t for t in all_tokens if t not in STD_SW and len(t) > 1]
fin_tokens    = [t for t in all_tokens if t not in FIN_SW  and len(t) > 1]

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.patch.set_facecolor("#0f0f1a")

configs = [
    ("No stop-word removal",         no_sw_tokens,  "#64748b"),
    ("Standard NLTK stop-words",     std_tokens,    "#ef4444"),
    ("Finance-adjusted stop-words",  fin_tokens,    "#22c55e"),
]

for ax, (label, tokens, color) in zip(axes, configs):
    freq = Counter(tokens).most_common(20)
    words, counts = zip(*freq)
    ax.barh(list(reversed(words)), list(reversed(counts)), color=color, alpha=0.85)
    ax.set_title(label, color="#ddd", fontsize=9, pad=6)
    ax.set_xlabel("Count")

plt.suptitle("Stop-word removal: three options — Enron email corpus",
             color="#ddd", y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

# Flag what is LOST with standard stop-words
lost = set(fin_tokens) - set(std_tokens)
print("Finance-significant tokens REMOVED by standard NLTK stop-word list:")
preserved_check = [t for t in FINANCE_PROTECTED if t in Counter(all_tokens)]
print("  Modal verbs removed:", [t for t in ["will","may","could","should","would","must"] if t in STD_SW])
print("  Negation removed:   ", [t for t in ["not","no","nor"] if t in STD_SW])
print("  Directional removed:", [t for t in ["up","down","above","below"] if t in STD_SW])

In [ ]:
# ── YOUR CHOICE ────────────────────────────────────────────────────────────
# Change the value below and re-run this cell.
# Options: "none" | "standard" | "finance"

STOPWORD_CHOICE = "finance"   # <--- YOUR CHOICE HERE

sw_map = {"none": set(), "standard": STD_SW, "finance": FIN_SW}
ACTIVE_SW = sw_map[STOPWORD_CHOICE]
print(f"Stop-word list set to: '{STOPWORD_CHOICE}'  ({len(ACTIVE_SW)} words removed)")

---
## Step 5: Number Handling

Numerical tokens introduce variance without necessarily adding linguistic information.
*"Revenue was $4.2bn"* and *"Revenue was $9.6bn"* should convey the same linguistic signal — the magnitude is captured in your quantitative data.

However, not all numbers are arbitrary:
- Regulatory references (*"Rule 10b-5"*)
- Percentage changes (*"grew 8%"*)
- Fiscal years (*"2001"*)

Choose how to handle numbers and see the effect on your token distribution.

In [ ]:
# ── YOUR CHOICE ────────────────────────────────────────────────────────────
# Options: "remove" | "retain" | "replace"

NUMBER_CHOICE = "remove"   # <--- YOUR CHOICE HERE

def handle_numbers(tokens, method):
    if method == "remove":
        return [t for t in tokens if not re.search(r'\d', t)]
    elif method == "replace":
        return ["NUM" if re.search(r'\d', t) else t for t in tokens]
    else:  # retain
        return tokens

# Apply current settings
all_tokens = get_tokens(corpus["body"].tolist(), lowercase=True, remove_punct=False)
filtered   = [t for t in all_tokens if t.isalpha() or re.search(r'\d', t)]
sw_filtered = [t for t in filtered if t not in ACTIVE_SW]
num_tokens = handle_numbers(sw_filtered, NUMBER_CHOICE)

nums_in_corpus = [t for t in sw_filtered if re.search(r'\d', t)]
print(f"Number handling: '{NUMBER_CHOICE}'")
print(f"Numeric tokens in corpus: {len(nums_in_corpus)}")
print(f"Examples: {nums_in_corpus[:20]}")
print(f"\nVocabulary size after number handling: {len(set(num_tokens)):,} unique tokens")

---
## Step 6: Stemming vs Lemmatisation

Both methods reduce words to a common form, but they work very differently.

**Stemming** (Porter, 1980) mechanically removes suffixes. Fast, but produces non-words:
- `liability` → `liabil`
- `rational` → `ration`

**Lemmatisation** reduces words to their canonical dictionary form:
- `liabilities` → `liability`
- `managing` → `manage`

The key consequence for financial text analysis: **dictionary matching**. The Loughran-McDonald lexicon contains `liability` but not `liabil`. Stemming silently destroys dictionary matches.

Run the comparison below, then make your choice.

In [ ]:
# Select 30 content-rich tokens from the corpus
sample_tokens = get_tokens(corpus["body"].tolist(), lowercase=True, remove_punct=True)
sample_tokens = [t for t in sample_tokens if t not in ACTIVE_SW and len(t) > 4]
top_tokens = [w for w, _ in Counter(sample_tokens).most_common(30)]

# Compare stem, lemma, and LM match
rows = []
for token in top_tokens:
    stem  = stemmer.stem(token)
    lemma = lemmatizer.lemmatize(token)
    lm_match_raw   = token in LM_NEGATIVE or token in LM_POSITIVE
    lm_match_stem  = stem  in LM_NEGATIVE or stem  in LM_POSITIVE
    lm_match_lemma = lemma in LM_NEGATIVE or lemma in LM_POSITIVE
    rows.append({"token": token, "stem": stem, "lemma": lemma,
                 "LM_raw": "✓" if lm_match_raw else "",
                 "LM_stem": "✓" if lm_match_stem else "",
                 "LM_lemma": "✓" if lm_match_lemma else ""})

df_norm = pd.DataFrame(rows)
print("Token normalisation comparison — top 30 content tokens")
print("(✓ = word found in Loughran-McDonald dictionary)")
print()
print(df_norm.to_string(index=False))
print()
raw_matches   = (df_norm["LM_raw"]   == "✓").sum()
stem_matches  = (df_norm["LM_stem"]  == "✓").sum()
lemma_matches = (df_norm["LM_lemma"] == "✓").sum()
print(f"LM dictionary matches — Raw: {raw_matches} | Stemmed: {stem_matches} | Lemmatised: {lemma_matches}")

In [ ]:
# ── YOUR CHOICE ────────────────────────────────────────────────────────────
# Options: "none" | "stemming" | "lemmatisation"

NORMALISATION_CHOICE = "lemmatisation"   # <--- YOUR CHOICE HERE

def normalise(tokens, method):
    if method == "stemming":
        return [stemmer.stem(t) for t in tokens]
    elif method == "lemmatisation":
        return [lemmatizer.lemmatize(t) for t in tokens]
    else:
        return tokens

print(f"Normalisation set to: '{NORMALISATION_CHOICE}'")

---
## Step 7: The Diagnostic — LM Dictionary Matching

This is where your pipeline choices become measurable.

Run the cell below to apply all your current settings to the full corpus and count how many tokens match the Loughran-McDonald financial sentiment dictionary. Then compare against a **deliberately misconfigured pipeline** (Porter stemming + standard NLTK stop-words) to see how many matches are lost.

*This directly illustrates the core argument of Renault (2020) and Loughran & McDonald (2011): pipeline choice matters more than algorithm choice.*

In [ ]:
def run_pipeline(texts, sw_set, number_method, norm_method):
    """Apply a full preprocessing pipeline and return token list."""
    all_tokens = []
    for text in texts:
        tokens = word_tokenize(text.lower())
        tokens = [t for t in tokens if t.isalpha() or re.search(r'\d', t)]
        tokens = handle_numbers(tokens, number_method)
        tokens = [t for t in tokens if t not in sw_set and len(t) > 1]
        tokens = normalise(tokens, norm_method)
        all_tokens.extend(tokens)
    return all_tokens

texts = corpus["body"].tolist()

# YOUR pipeline
your_tokens = run_pipeline(texts, ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)

# Misconfigured pipeline (lecture diagnostic)
bad_tokens  = run_pipeline(texts, STD_SW, "retain", "stemming")

def count_lm_matches(tokens):
    neg = sum(1 for t in tokens if t in LM_NEGATIVE)
    pos = sum(1 for t in tokens if t in LM_POSITIVE)
    return pos, neg

your_pos, your_neg = count_lm_matches(your_tokens)
bad_pos,  bad_neg  = count_lm_matches(bad_tokens)

# Results table
print("=" * 60)
print("PIPELINE DIAGNOSTIC — LM DICTIONARY MATCHING")
print("=" * 60)
print(f"{'':30s} {'Your pipeline':>15} {'Misconfigured':>14}")
print("-" * 60)
print(f"{'Vocabulary (unique tokens)':30s} {len(set(your_tokens)):>15,} {len(set(bad_tokens)):>14,}")
print(f"{'Total tokens':30s} {len(your_tokens):>15,} {len(bad_tokens):>14,}")
print(f"{'LM Positive matches':30s} {your_pos:>15,} {bad_pos:>14,}")
print(f"{'LM Negative matches':30s} {your_neg:>15,} {bad_neg:>14,}")
print(f"{'Total LM matches':30s} {your_pos+your_neg:>15,} {bad_pos+bad_neg:>14,}")
print("=" * 60)

loss_pct = ((your_pos+your_neg) - (bad_pos+bad_neg)) / max(your_pos+your_neg, 1) * 100
print(f"\nYour pipeline captures {abs(loss_pct):.1f}% {'more' if loss_pct>0 else 'fewer'} LM matches than the misconfigured version.")
print("\nYour pipeline settings:")
print(f"  Stop-words:     {STOPWORD_CHOICE}")
print(f"  Numbers:        {NUMBER_CHOICE}")
print(f"  Normalisation:  {NORMALISATION_CHOICE}")

In [ ]:
# Visualise LM matches by email period
period_results = []
for period, grp in corpus.groupby("period"):
    tokens = run_pipeline(grp["body"].tolist(), ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)
    pos, neg = count_lm_matches(tokens)
    total_words = grp["word_count"].sum()
    period_results.append({
        "period": period,
        "positive_rate": pos / total_words * 1000,
        "negative_rate": neg / total_words * 1000,
    })

df_res = pd.DataFrame(period_results)
order  = ["pre_crisis", "crisis", "collapse"]
df_res = df_res.set_index("period").reindex(order)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor("#0f0f1a")
x = np.arange(len(order))
w = 0.35
ax.bar(x - w/2, df_res["positive_rate"], w, color="#22c55e", alpha=0.85, label="LM Positive")
ax.bar(x + w/2, df_res["negative_rate"], w, color="#ef4444", alpha=0.85, label="LM Negative")
ax.set_xticks(x)
ax.set_xticklabels(["Pre-crisis\n(2000)", "Crisis\n(2001)", "Collapse\n(2002)"])
ax.set_ylabel("Matches per 1,000 words")
ax.set_title("LM Sentiment Signal by Period — Enron Email Corpus\n(Your current pipeline settings)",
             color="#ddd", pad=10)
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 8: Pruning

Pruning removes tokens that appear too rarely or too frequently across the corpus. The goal is to reduce dimensionality — removing tokens that carry little discriminatory information.

- **Minimum frequency (min_df):** remove tokens appearing in fewer than N emails
- **Maximum frequency (max_df):** remove tokens appearing in more than X% of emails

Adjust the thresholds below and observe how vocabulary size changes.

In [ ]:
# ── YOUR CHOICES ───────────────────────────────────────────────────────────
MIN_DF = 2    # remove tokens appearing in fewer than this many emails
MAX_DF = 0.9  # remove tokens appearing in more than this fraction of emails

# Build document-level token sets
all_tokens = run_pipeline(corpus["body"].tolist(), ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)
doc_tokens  = [
    run_pipeline([body], ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)
    for body in corpus["body"]
]

# Document frequency of each token
n_docs = len(doc_tokens)
df_counts = Counter()
for doc in doc_tokens:
    for token in set(doc):
        df_counts[token] += 1

# Pruning
full_vocab = set(all_tokens)
after_min  = {t for t in full_vocab if df_counts[t] >= MIN_DF}
after_both = {t for t in after_min  if df_counts[t] / n_docs <= MAX_DF}

print(f"Vocabulary sizes:")
print(f"  Full (no pruning):           {len(full_vocab):>6,} unique tokens")
print(f"  After min_df >= {MIN_DF}:          {len(after_min):>6,} unique tokens")
print(f"  After min_df + max_df <= {int(MAX_DF*100)}%:  {len(after_both):>6,} unique tokens")
print(f"  Tokens removed by pruning:   {len(full_vocab) - len(after_both):>6,}")
print()

# What does max_df remove? (near-universal boilerplate)
boilerplate = sorted(
    [(t, df_counts[t]/n_docs) for t in full_vocab if df_counts[t]/n_docs > MAX_DF],
    key=lambda x: -x[1]
)[:20]
print(f"Tokens appearing in > {int(MAX_DF*100)}% of emails (boilerplate):")
for t, pct in boilerplate:
    print(f"  {t:<20} {pct:.0%}")

---
## Step 9: Comparing the Two Dictionaries

The lecture highlighted that **73–80% of negative words identified by the Harvard General Inquirer are not negative in a financial context** (Loughran & McDonald, 2011).

The cell below applies both dictionaries to your pipeline output so you can see the difference in practice.

In [ ]:
texts = corpus["body"].tolist()
your_tokens = run_pipeline(texts, ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)
token_set   = set(your_tokens)

# Find words flagged by Harvard but not by LM — the false positives
hv_neg_not_lm  = sorted(token_set & HV_NEGATIVE - LM_NEGATIVE)[:25]
hv_pos_not_lm  = sorted(token_set & HV_POSITIVE - LM_POSITIVE)[:25]
lm_neg_not_hv  = sorted(token_set & LM_NEGATIVE - HV_NEGATIVE)[:25]

print("Tokens flagged NEGATIVE by Harvard GI but NOT by LM (likely false positives):")
print("  ", hv_neg_not_lm)
print()
print("Tokens flagged NEGATIVE by LM but NOT by Harvard GI (finance-specific negatives):")
print("  ", lm_neg_not_hv)
print()

# Side-by-side match counts
lm_pos, lm_neg   = count_lm_matches(your_tokens)
hv_pos = sum(1 for t in your_tokens if t in HV_POSITIVE)
hv_neg = sum(1 for t in your_tokens if t in HV_NEGATIVE)

fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor("#0f0f1a")
x = np.arange(2)
w = 0.3
ax.bar(x - w/2, [lm_pos, lm_neg], w, color=["#22c55e","#ef4444"], alpha=0.85,
       label="Loughran-McDonald")
ax.bar(x + w/2, [hv_pos, hv_neg], w, color=["#86efac","#fca5a5"], alpha=0.85,
       label="Harvard General Inquirer")
ax.set_xticks(x)
ax.set_xticklabels(["Positive matches", "Negative matches"])
ax.set_title("Dictionary comparison — same pipeline, different lexicons", color="#ddd", pad=10)
ax.legend()
ax.set_ylabel("Total matches")
plt.tight_layout()
plt.show()

---
## Step 10: Bias in This Corpus

The lecture covered six forms of corpus bias (Section 6.5, Grimmer, Roberts & Stewart, 2022):

| Bias | Definition |
|---|---|
| **Resource** | Texts over-represent those with resources to produce them |
| **Incentive** | Documents shaped by strategic incentives of the author |
| **Medium** | Platform constraints shape language in ways that differ by source |
| **Retrieval** | Corpus reflects the retrieval logic as much as the underlying population |
| **Survivorship** | Only entities that persisted are included |
| **Self-reporting** | Authors chose what to record and how to present it |

Consider this Enron corpus in light of each of these biases. Use **Worksheet Questions 6–7** to record your thinking.

In [ ]:
# Illustrate self-reporting and incentive bias:
# How does the sentiment signal change across the three periods?

period_sentiment = {}
for period, grp in corpus.groupby("period"):
    tokens = run_pipeline(grp["body"].tolist(), ACTIVE_SW, NUMBER_CHOICE, NORMALISATION_CHOICE)
    pos, neg = count_lm_matches(tokens)
    total = len(tokens)
    period_sentiment[period] = {
        "positive_rate": pos / total if total > 0 else 0,
        "negative_rate": neg / total if total > 0 else 0,
        "net_sentiment": (pos - neg) / total if total > 0 else 0,
    }

order = ["pre_crisis", "crisis", "collapse"]
labels = ["Pre-crisis
(2000)", "Crisis
(2001)", "Collapse
(2002)"]
net = [period_sentiment[p]["net_sentiment"] * 100 for p in order]

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor("#0f0f1a")
colors = ["#22c55e" if v >= 0 else "#ef4444" for v in net]
bars = ax.bar(labels, net, color=colors, alpha=0.85, width=0.5)
ax.axhline(0, color="#666", linewidth=0.8)
ax.set_ylabel("Net LM sentiment (% of tokens)")
ax.set_title("Net LM sentiment across periods — does this reflect what actually happened?\n"
             "Consider: incentive bias, self-reporting bias", color="#ddd", pad=10)
for bar, val in zip(bars, net):
    ax.text(bar.get_x() + bar.get_width()/2, val + (0.01 if val >= 0 else -0.02),
            f"{val:+.2f}%", ha="center", va="bottom" if val >= 0 else "top",
            color="#ddd", fontsize=9)
plt.tight_layout()
plt.show()

print("\nConsider: The emails were written by executives and managers.")
print("Does the sentiment score reflect the true condition of the company,")
print("or the authors' strategic incentives to manage impressions?")

---
## Summary: Your Pipeline Choices

Record your final pipeline settings below and use them to complete the worksheet.

| Decision | Your choice | Justification |
|---|---|---|
| Tokenisation | NLTK word_tokenize | |
| Case folding | Yes | |
| Punctuation | Removed (alpha only) | |
| Stop-words | `STOPWORD_CHOICE` | |
| Number handling | `NUMBER_CHOICE` | |
| Normalisation | `NORMALISATION_CHOICE` | |
| Min document frequency | `MIN_DF` | |
| Max document frequency | `MAX_DF` | |

**References**

Grimmer, J., Roberts, M.E. and Stewart, B.M. (2022) *Text as Data*. Princeton University Press.

Klimt, B. and Yang, Y. (2004) 'The Enron Corpus', *Machine Learning: ECML 2004*, pp. 217–226.

Loughran, T. and McDonald, B. (2011) 'When is a Liability not a Liability?', *Journal of Finance*, 66(1), pp. 35–65.

Renault, T. (2020) 'Sentiment analysis and machine learning in finance', *Digital Finance*, 2, pp. 1–13.

Todd, A., Bowden, J. and Moshfeghi, Y. (2024) 'Text-based sentiment analysis in finance', *Intelligent Systems in Accounting, Finance and Management*, 31.